# 🛡️ Cybersecurity Incident Type Classifier
## Deep Learning — Text Classification with MLP Neural Network

**Objective:** Automatically classify cybersecurity incidents into attack types  
(Malware, Ransomware, Phishing, Data Breach, etc.) from incident titles and summaries.

**Dataset:** Combined `global_news_rows.csv` + `incidents_rows.csv` (237 incidents)  
**Model:** TF-IDF Vectoriser → Multi-Layer Perceptron (MLP) Neural Network  
**Alignment:** Malaysia NAIO Action Plan 2026–2030 — Area 3 (AI Adaptation), Area 5 (AI Impact Study)


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.neural_network import MLPClassifier

# Reproducibility
np.random.seed(42)
print("Libraries loaded successfully.")


## 2. Load & Combine Datasets

In [ ]:
gn  = pd.read_csv('global_news_rows.csv')
inc = pd.read_csv('incidents_rows.csv')

df = pd.concat([gn, inc], ignore_index=True)
print(f"global_news  : {len(gn)} rows")
print(f"incidents    : {len(inc)} rows")
print(f"Combined     : {len(df)} rows")
df.head(3)


## 3. Exploratory Data Analysis

In [ ]:
print("Columns:", df.columns.tolist())
print(f"\nNull counts:\n{df[['title','summary','incident_type','category','country']].isnull().sum()}")


In [ ]:
# Class distribution
print("incident_type distribution:")
print(df['incident_type'].value_counts())


In [ ]:
# Visualise class distribution
plt.figure(figsize=(12, 5))
df['incident_type'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Incident Type Distribution (Before Cleaning)', fontsize=14)
plt.xlabel('Incident Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Country distribution
plt.figure(figsize=(12, 4))
df['country'].value_counts().head(15).plot(kind='bar', color='coral', edgecolor='black')
plt.title('Top 15 Countries', fontsize=14)
plt.xlabel('Country')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Category distribution
plt.figure(figsize=(12, 4))
df['category'].value_counts().plot(kind='bar', color='mediumseagreen', edgecolor='black')
plt.title('Incident Category Distribution', fontsize=14)
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 4. Data Preprocessing

In [ ]:
# Combine title + summary into a single text feature
df['text'] = df['title'].fillna('') + ' ' + df['summary'].fillna('')

# Drop rows with no incident_type label
df = df.dropna(subset=['incident_type'])
df = df[df['incident_type'].str.strip() != '']

# Consolidate near-duplicate class names
label_map = {
    'Ransomware Attack'  : 'Ransomware',
    'Credential Stuffing': 'Data Breach',
    'Multiple'           : 'Others',
    'APT'                : 'Advanced Persistent Threat (APT)',
    'Unauthorised Access': 'Others',
}
df['incident_type'] = df['incident_type'].replace(label_map)

# Keep only classes with >= 4 samples (needed for stratified split)
counts = df['incident_type'].value_counts()
valid  = counts[counts >= 4].index
df     = df[df['incident_type'].isin(valid)].reset_index(drop=True)

print(f"Final dataset: {len(df)} rows, {df['incident_type'].nunique()} classes")
print()
print("Final class distribution:")
print(df['incident_type'].value_counts())


In [ ]:
# Visualise final class distribution
plt.figure(figsize=(12, 5))
df['incident_type'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Incident Type Distribution (After Cleaning)', fontsize=14)
plt.xlabel('Incident Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 5. Feature Engineering — TF-IDF Vectorisation

In [ ]:
# Encode labels to integers
le = LabelEncoder()
y  = le.fit_transform(df['incident_type'])

print("Classes:", le.classes_)
print("Encoded:", np.unique(y))


In [ ]:
# Train / Test split  (80% train, 20% test — stratified)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'], y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y
)

print(f"Train : {len(X_train_text)} samples")
print(f"Test  : {len(X_test_text)} samples")


In [ ]:
# TF-IDF: fit ONLY on training text, then transform both splits
#   max_features=3000  — vocabulary size
#   ngram_range=(1,2)  — unigrams + bigrams
#   sublinear_tf=True  — apply log(1+tf) to reduce impact of very frequent words

vectoriser = TfidfVectorizer(
    max_features = 3000,
    ngram_range  = (1, 2),
    sublinear_tf = True,
    strip_accents = 'unicode',
    stop_words   = 'english',
)

X_train = vectoriser.fit_transform(X_train_text)
X_test  = vectoriser.transform(X_test_text)

print(f"TF-IDF feature matrix — Train: {X_train.shape}, Test: {X_test.shape}")


## 6. Build & Train MLP Neural Network

**Architecture:**
```
TF-IDF features (3000)
       ↓
 Dense Layer 1: 256 neurons, ReLU activation
       ↓
 Dropout: 0.3
       ↓
 Dense Layer 2: 128 neurons, ReLU activation
       ↓
 Dropout: 0.3
       ↓
 Dense Layer 3: 64 neurons, ReLU activation
       ↓
 Output: 13 classes (Softmax)
```
**Optimiser:** Adam  |  **Loss:** Cross-Entropy  |  **Regularisation:** L2 + Early Stopping


In [ ]:
# Build MLP classifier
# hidden_layer_sizes = (256, 128, 64) => 3 hidden layers
# activation = 'relu'
# solver = 'adam'
# alpha = 0.001  => L2 regularisation to prevent overfitting
# early_stopping = True => holds out 15% of training data as internal validation
# n_iter_no_change = 15 => stops if val score doesn't improve for 15 epochs

mlp = MLPClassifier(
    hidden_layer_sizes = (256, 128, 64),
    activation         = 'relu',
    solver             = 'adam',
    alpha              = 0.001,
    max_iter           = 500,
    random_state       = 42,
    early_stopping     = True,
    validation_fraction= 0.15,
    n_iter_no_change   = 15,
    verbose            = False,
)

mlp.fit(X_train, y_train)
print(f"Training complete — ran for {mlp.n_iter_} epochs (early stopping active)")


## 7. Model Evaluation

In [ ]:
y_pred = mlp.predict(X_test)

accuracy  = accuracy_score(y_test, y_pred)
f1_macro  = f1_score(y_test, y_pred, average='macro',    zero_division=0)
f1_weight = f1_score(y_test, y_pred, average='weighted', zero_division=0)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_test, y_pred, average='weighted',    zero_division=0)

print("=" * 50)
print("       MLP CLASSIFIER EVALUATION")
print("=" * 50)
print(f"  Accuracy           : {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"  Precision (wtd)    : {precision:.4f}")
print(f"  Recall    (wtd)    : {recall:.4f}")
print(f"  F1 Score  (wtd)    : {f1_weight:.4f}")
print(f"  F1 Score  (macro)  : {f1_macro:.4f}")
print("=" * 50)
print()
print("Per-class Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))


In [ ]:
# Confusion Matrix
plt.figure(figsize=(13, 10))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm,
    annot      = True,
    fmt        = 'd',
    cmap       = 'Blues',
    xticklabels= le.classes_,
    yticklabels= le.classes_,
    linewidths = 0.5,
)
plt.title('Confusion Matrix — MLP Classifier', fontsize=14, pad=15)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Per-class F1 bar chart
report_dict = classification_report(
    y_test, y_pred, target_names=le.classes_,
    output_dict=True, zero_division=0
)
class_f1 = {k: v['f1-score'] for k, v in report_dict.items() if k in le.classes_}

plt.figure(figsize=(12, 5))
bars = plt.bar(class_f1.keys(), class_f1.values(), color='steelblue', edgecolor='black')
plt.axhline(y=f1_weight, color='red', linestyle='--', label=f'Weighted F1 = {f1_weight:.2f}')
plt.title('F1 Score per Incident Type', fontsize=14)
plt.xlabel('Incident Type')
plt.ylabel('F1 Score')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.ylim(0, 1.1)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Training loss curve
plt.figure(figsize=(10, 4))
plt.plot(mlp.loss_curve_, label='Training Loss', color='steelblue')
if mlp.validation_scores_ is not None:
    # Convert validation accuracy to loss proxy (1 - acc)
    val_loss_proxy = [1 - v for v in mlp.validation_scores_]
    plt.plot(val_loss_proxy, label='Validation Loss (1 - acc)', color='coral', linestyle='--')
plt.title('MLP Training Loss Curve', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Evaluation Summary Card

In [ ]:
print()
print("╔══════════════════════════════════════════════════════╗")
print("║         MLP INCIDENT CLASSIFIER — EVALUATION CARD   ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Model            : MLP Neural Network               ║")
print(f"║  Architecture     : TF-IDF → 256 → 128 → 64 → 13    ║")
print(f"║  Activation       : ReLU                             ║")
print(f"║  Optimiser        : Adam                             ║")
print(f"║  Regularisation   : L2 (alpha=0.001) + Early Stop    ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Dataset          : 237 incidents, 13 classes        ║")
print(f"║  Train / Test     : 80% / 20%  (stratified)          ║")
print(f"║  Train samples    : {len(X_train_text):<5}                             ║")
print(f"║  Test  samples    : {len(X_test_text):<5}                             ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Accuracy         : {accuracy*100:>6.2f}%                          ║")
print(f"║  Precision (wtd)  : {precision:>6.4f}                          ║")
print(f"║  Recall    (wtd)  : {recall:>6.4f}                          ║")
print(f"║  F1 Score  (wtd)  : {f1_weight:>6.4f}                          ║")
print(f"║  F1 Score  (macro): {f1_macro:>6.4f}                          ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Epochs run       : {mlp.n_iter_:<5} (early stopping)           ║")
print("╚══════════════════════════════════════════════════════╝")


## 9. Live Prediction Demo

Demonstrate the classifier on new, unseen incident headlines —  
simulating how the CSM dashboard would auto-tag incoming threats.


In [ ]:
def classify_incident(text):
    vec   = vectoriser.transform([text])
    pred  = mlp.predict(vec)[0]
    proba = mlp.predict_proba(vec)[0]
    label = le.inverse_transform([pred])[0]
    conf  = proba.max() * 100
    top3  = sorted(zip(le.classes_, proba), key=lambda x: -x[1])[:3]
    print(f"Input  : {text[:80]}...")
    print(f"Prediction : {label}  (confidence: {conf:.1f}%)")
    print("Top 3  :")
    for cls, p in top3:
        print(f"   {cls:<40} {p*100:.1f}%")
    print()

# Test with real-world style headlines
classify_incident(
    "CIMB Bank Malaysia suffers data breach exposing customer financial records on dark web"
)
classify_incident(
    "LockBit ransomware group claims attack on Malaysian government ministry, demands payment"
)
classify_incident(
    "Critical zero-day vulnerability discovered in Microsoft Exchange affecting all versions"
)
classify_incident(
    "Phishing campaign targets Maybank users via fake SMS login pages stealing credentials"
)
classify_incident(
    "APT group linked to nation-state deploys backdoor malware in Southeast Asian telecoms"
)


## 10. Top TF-IDF Features per Class

In [ ]:
# Show the most influential words for each class
feature_names = np.array(vectoriser.get_feature_names_out())
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i, cls in enumerate(le.classes_):
    # Get weights of output layer neuron for this class
    # MLP stores coefs_ for each layer; last layer maps hidden→output
    # We use the dot product of all weight matrices as a proxy for feature importance
    # Simpler: use the absolute weight from first layer × mean of subsequent layers
    w = np.abs(mlp.coefs_[0]).mean(axis=1)  # (n_features,) — avg abs weight to first hidden layer
    top_idx   = w.argsort()[-15:][::-1]
    top_words = feature_names[top_idx]
    top_vals  = w[top_idx]

    axes[i].barh(top_words[::-1], top_vals[::-1], color='steelblue')
    axes[i].set_title(cls, fontsize=9, fontweight='bold')
    axes[i].tick_params(labelsize=7)

# Hide unused subplots
for j in range(len(le.classes_), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Top TF-IDF Feature Weights (First Layer) per Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 11. Save Model Artefacts (for Dashboard Integration)

In [ ]:
import pickle

with open('mlp_classifier.pkl', 'wb') as f:
    pickle.dump(mlp, f)

with open('tfidf_vectoriser.pkl', 'wb') as f:
    pickle.dump(vectoriser, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Saved:")
print("  mlp_classifier.pkl")
print("  tfidf_vectoriser.pkl")
print("  label_encoder.pkl")
print()
print("Load in dashboard with:")
print("  import pickle")
print("  mlp = pickle.load(open('mlp_classifier.pkl','rb'))")
print("  vec = pickle.load(open('tfidf_vectoriser.pkl','rb'))")
print("  le  = pickle.load(open('label_encoder.pkl','rb'))")
